### Colab Mount


In [ ]:
import os
import sys
from pathlib import Path

try:
    import google.colab
    from google.colab import drive

    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")

src_dir = Path("/content/drive/MyDrive/Research/AI-In-Science-Lab/src/local_learning")
if not (src_dir / "training.py").is_file() or not (src_dir / "models").is_dir():
    raise FileNotFoundError(f"Expected local_learning source directory at {src_dir}.")

os.chdir(src_dir)
sys.path = [path for path in sys.path if path != str(src_dir)]
sys.path.insert(0, str(src_dir))
for module_name in list(sys.modules):
    if module_name in {"training", "datasets", "models", "losses", "loggers"}:
        del sys.modules[module_name]
    elif module_name.startswith(("datasets.", "models.", "losses.", "loggers.")):
        del sys.modules[module_name]
print(f"[src] {src_dir}")


### Setup


In [ ]:
import torch
import wandb

from training import train

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[device] {DEVICE}")


### Experiment Configs


In [ ]:
VGG16_BASE_CONFIG = {
    "vgg16_first_full_training_epoch": None,
    "vgg16_deconv_training": False,
    "vgg16_local_training": False,
    "vgg16_small": True,
}

GSA_BASE_CONFIG = {
    "gsa_hidden_channels": [64, 48, 32],
    "gsa_local_training": True,
    "gsa_local_reconstruction_strength": 1.0,
}

BASE_CONFIG = {
    **GSA_BASE_CONFIG,
    "project": "local-learning-6",
    "architecture_type": "greedy_stacked_autoencoder",
    "data": "cifar10:normalize",
    "loss_type": "regular",
    "learning_rate": 3e-4,
    "epochs": 20,
    "batch_size": 64,
    "num_workers": 2,
    "lambda_strength": 0.0,
    "kernel_size": 3,
    "correlation_loss": "sum",
    "k_spatial": None,
    "k_population": None,
    "k_lifetime": 0.1,
    "k_population_alpha": 1.0,
    "log_every_steps": 100,
}


### Sweep


In [ ]:
SWEEP_PROJECT = "greedy-stacked-autoencoders"
SWEEP_BASE_CONFIG = {**BASE_CONFIG, "project": SWEEP_PROJECT}

SWEEP_VALUES = {
    "learning_rate": [1e-4],
    "gsa_local_training": [False, True],
    "gsa_local_reconstruction_strength": [0.0, 1.0],
    "run_repeat": [0, 1],
}


def wandb_sweep_parameters(base_config: dict, sweep_values: dict) -> dict:
    parameters = {key: {"value": value} for key, value in base_config.items()}
    for key, values in sweep_values.items():
        parameters[key] = {"values": values if isinstance(values, list) else list(values)}
    return parameters


SWEEP_CONFIG = {
    "name": "gsa-cifar10-local-training-reconstruction-grid",
    "project": SWEEP_PROJECT,
    "method": "grid",
    "metric": {
        "name": "test/acc",
        "goal": "maximize",
    },
    "parameters": wandb_sweep_parameters(SWEEP_BASE_CONFIG, SWEEP_VALUES),
}


def sweep_train() -> None:
    train(SWEEP_BASE_CONFIG, device=DEVICE)


def run_sweep(*, sweep_id: str | None = None) -> str:
    if sweep_id is None:
        sweep_id = wandb.sweep(SWEEP_CONFIG, project=SWEEP_PROJECT)

    wandb.agent(sweep_id, function=sweep_train)
    return sweep_id


### Run


In [ ]:
sweep_id = run_sweep()
sweep_id
